# AI Resume Analyzer — Core Notebook
This notebook demonstrates the core parsing, skill extraction and role-ranking pipeline. The Streamlit app is in `app.py`.

In [ ]:
!pip -q install pypdf python-docx scikit-learn pandas


In [ ]:
import re, io, pandas as pd
from pypdf import PdfReader
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
ROLE_SKILLS = {
 "Data Analyst": ["Python","SQL","Excel","Pandas","Power BI","Tableau","Statistics","NumPy"],
 "Machine Learning Engineer": ["Python","Machine Learning","scikit-learn","Pandas","NumPy","TensorFlow","PyTorch","FastAPI","Docker","SQL"],
 "AI Engineer": ["Python","Deep Learning","LLM","RAG","APIs","PyTorch","TensorFlow","NLP","Transformers","Docker"],
 "NLP Engineer": ["Python","NLP","Transformers","Hugging Face","spaCy","Machine Learning","PyTorch","LLM","Text Classification"],
 "Computer Vision Engineer": ["Python","OpenCV","CNN","YOLO","Deep Learning","PyTorch","TensorFlow","Image Processing"],
 "Python Developer": ["Python","OOP","Git","REST API","FastAPI","Django","Flask","SQL","Testing"],
 "Data Engineer": ["Python","SQL","ETL","Spark","Hadoop","Airflow","AWS","Docker","Data Warehousing"],
 "Web Developer": ["HTML","CSS","JavaScript","React","Node.js","REST API","Git","SQL"]}
SKILLS = sorted(set(s for v in ROLE_SKILLS.values() for s in v) | {"Java","C","C++","JavaScript","MongoDB","MySQL","PostgreSQL","AWS","Azure","Linux","Kubernetes","GitHub","Jupyter","Matplotlib","Seaborn","Keras","Flask","Django","HTML","CSS","React","Node.js","Computer Networks","DBMS","Data Structures","Algorithms","Agile","Communication","Problem Solving"},key=len,reverse=True)


In [ ]:
def extract_resume_text(file_bytes, filename):
    if filename.lower().endswith('.pdf'):
        return '\n'.join((p.extract_text() or '') for p in PdfReader(io.BytesIO(file_bytes)).pages)
    if filename.lower().endswith('.docx'):
        return '\n'.join(p.text for p in Document(io.BytesIO(file_bytes)).paragraphs)
    raise ValueError('Upload a PDF or DOCX file')

def clean_text(text):
    return re.sub(r'\s+',' ',text.lower()).strip()

def extract_skills(text):
    t=clean_text(text); found=[]
    for skill in SKILLS:
        if re.search(r'(?<![a-z0-9])'+re.escape(skill.lower())+r'(?![a-z0-9])',t): found.append(skill)
    return sorted(set(found),key=str.lower)

def rank_roles(text):
    corpus=[text]+[' '.join(v) for v in ROLE_SKILLS.values()]
    X=TfidfVectorizer(ngram_range=(1,2),stop_words='english').fit_transform(corpus)
    sims=cosine_similarity(X[:1],X[1:]).ravel()
    return sorted(zip(ROLE_SKILLS.keys(),sims),key=lambda x:x[1],reverse=True)


In [ ]:
# Demo text (replace this with extracted text from your own resume)
sample_resume = "Python SQL Pandas NumPy machine learning scikit-learn data analysis Excel Power BI Git project"
print('Skills detected:', extract_skills(sample_resume))
print('Top 3 roles:')
for role, score in rank_roles(clean_text(sample_resume))[:3]:
    print(f'{role}: {score*100:.1f}%')


In [ ]:
# Optional: upload a resume in Colab and run the pipeline
from google.colab import files
uploaded = files.upload()  # Select a PDF or DOCX resume
filename = next(iter(uploaded))
resume_text = extract_resume_text(uploaded[filename], filename)
print(resume_text[:1200])
print('Detected skills:', extract_skills(resume_text))
print('Ranked roles:', [(r, round(s*100,1)) for r,s in rank_roles(clean_text(resume_text))])
